In [ ]:
# See preprocessing.ipynb for Figures S1, S3 and S5

In [1]:
import scanpy as sc
import harmonypy as hm
import tqdm as notebook_tqdm
import phate
import pandas as pd
from matplotlib.colors import to_rgba
from pygam import LinearGAM, s
import meld
import scipy.stats as stats
import numpy as np
import sklearn as sk
import matplotlib.pyplot as plt
import matplotlib
from matplotlib import pyplot as plt
import seaborn as sns
from itertools import combinations
import gseapy as gp
from scipy.stats import ttest_ind
from statannotations.Annotator import Annotator
import scprep

In [2]:
adata= sc.read_h5ad("/Users/Jan/data/final_final_data.h5ad")

In [ ]:
# Figure S2
adata_ka= adata[adata.obs['Sample Type']== 'KA']
vmin, vmax= 0, 1
timepoints= adata_ka.obs['Sample name'].unique()
pairs= [(timepoints[0], timepoints[2]), (timepoints[1], timepoints[0]), (timepoints[1], timepoints[2])]
for group_a, group_b in pairs:
    pair_mask= adata_ka.obs['Sample name'].isin([group_a, group_b])
    subset_pair= adata_ka[pair_mask].copy()
    try:
        meld_op = meld.MELD()
        sample_densities= meld_op.fit_transform(subset_pair, sample_labels=subset_pair.obs['Sample name'])
        sample_densities= sk.preprocessing.normalize(
            sample_densities,
            norm='l1',
            axis=1
        )
        cols= sorted([group_a, group_b])
        densities_df = pd.DataFrame(sample_densities, index=subset_pair.obs_names, columns=cols)
        col_name= f"meld_{group_a}_vs_{group_b}"
        adata_ka.obs[col_name]= np.nan
        adata_ka.obs.loc[subset_pair.obs_names, col_name]= densities_df[group_b]
        sc.pl.embedding(
            adata_ka, 
            basis='phate', 
            color=col_name,
            cmap='magma',
            title=f"{group_a} vs {group_b} MELD density",
            frameon=False,
            na_color= '#e0e0e0',
            vmin= vmin,
            vmax= vmax,
            show= True,
            save=f"_{group_a}, {group_b}_meld"
        )
    except Exception as e:
        print(f"Error: {e}")

In [ ]:
# Figure S4
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.set_figure_params(dpi=100, color_map = "viridis", frameon=True, transparent=True,
                    dpi_save=800, facecolor="None", format="pdf", figsize=[4,4])
gene_signatures= pd.read_csv("C:/Users/Jan/data/CuratedEpithelia_geneSet.csv")
csc_genes= gene_signatures["GENE"][gene_signatures["ANNOTATION"]== "CSC"].tolist()
procsc_genes= gene_signatures["GENE"][gene_signatures["ANNOTATION"]== "proCSC"].tolist()
revcsc_genes= gene_signatures["GENE"][gene_signatures["ANNOTATION"]== "revCSC"].tolist()
replication_genes= gene_signatures["GENE"][gene_signatures["ANNOTATION"]== "Replication"].tolist()
erstress_genes= gene_signatures["GENE"][gene_signatures["ANNOTATION"]== "ER stress"].tolist()
secretory_genes= gene_signatures["GENE"][gene_signatures["ANNOTATION"]== "Secretory"].tolist()
absorptive_genes= gene_signatures["GENE"][gene_signatures["ANNOTATION"]== "Absorptive"].tolist()
sc.tl.score_genes(adata, csc_genes, score_name= 'CSC_score')
sc.tl.score_genes(adata, procsc_genes, score_name='proCSC_score')
sc.tl.score_genes(adata, revcsc_genes, score_name='revCSC_score')
sc.tl.score_genes(adata, replication_genes, score_name='Replication_score')
sc.tl.score_genes(adata, erstress_genes, score_name='ER_stress_score')
sc.tl.score_genes(adata, secretory_genes, score_name='Secretory_score')
sc.tl.score_genes(adata, absorptive_genes, score_name='Absorptive_score')
cell_signatures= ["CSC_score", 'proCSC_score', 'revCSC_score', 'Replication_score', 'ER_stress_score', 'Secretory_score', 'Absorptive_score']
title_names = [s.replace('_score', '') for s in cell_signatures]
vmin= np.min([adata.obs[cell].min() for cell in cell_signatures])
vmax= np.max([adata.obs[cell].max() for cell in cell_signatures])
for i, cell_signature in enumerate(cell_signatures):
    sc.external.pl.phate(adata, color= cell_signature, cmap= "Reds", title= f"{title_names[i]}", frameon= False, vmin= vmin, vmax= vmax)